# REX3 EXIOBASE walkthrough

This notebook is the practical walkthrough for parsing the monetary EXIOBASE REX3 bundle in MARIO.

It focuses on:

- the complete REX3 input-output table;
- automatic download of the economic and environmental bundles;
- the detailed greenhouse-gas extension;
- simple checks of the parsed database.

See the [web guide](rex3.rst) for folder layouts, source releases and numerical assumptions.

The pinned REX3 releases use CC BY 4.0. Retain author attribution and indicate changes.


## What this notebook covers

- which public parser entry points to use;
- which Zenodo records are downloaded;
- what local directory layout MARIO expects;
- how to download and parse one REX3 year;
- how the detailed greenhouse-gas rows are imported;
- how to inspect totals by country and sector.

## Relevant Zenodo releases

The REX3 workflow combines two official records:

- [REX3 monetary MRIO tables and labels](https://zenodo.org/records/10354283), covering 1995–2022;
- [REX3 greenhouse-gas extension](https://zenodo.org/records/14724086), covering 1995–2022.

The labels and greenhouse-gas companion are always required and are therefore included automatically by the downloader.

## Main entry points

For normal user workflows, use the generic EXIOBASE dispatcher:

- `mario.parse_exiobase(..., variant="REX3")`.

The same parser is also available directly as `mario.parse_exiobase_rex(...)`. To download without parsing, use `mario.download_exiobase_rex(...)`.

## Key arguments

The key public arguments are:

- `path`: local REX3 root or download/cache directory;
- `table`: use `"IOT"`;
- `unit`: use `"Monetary"` with the generic dispatcher;
- `variant`: use `"REX3"` with the generic dispatcher;
- `year`: select one year from 1995 through 2022;
- `download`: download missing files before parsing;
- `overwrite_download`: replace previously downloaded files;
- `labels_path` and `ghg_path`: optional explicit companion directories for a manually arranged local dataset;
- `calc_all`: calculate derived MARIO matrices immediately after parsing.

## Local layout expectation

When `download=True`, `path` is the destination/cache directory. MARIO creates and reuses a layout like this:

```text
rex3/
├── REX3_2022/
│   ├── T_REX3.mat
│   ├── Y_REX3.mat
│   ├── Q_REX3.mat
│   └── Q_Y_REX3.mat
├── REX3_Labels/
│   ├── Labels_Countries_REX3.mat
│   ├── Labels_Sectors_REX3.mat
│   ├── Labels_T_REX3.mat
│   └── Labels_Q_REX3.mat
└── GHG_data_REX3/
    ├── d_CC_REX3_timeline.mat
    ├── Q_LULUCF_REX3_timeline.mat
    ├── Q_Y_LULUCF_REX3_timeline.mat
    └── Labels/
```

For a manual local workflow, `path` can point to the common root or directly to one `REX3_<year>` directory. MARIO auto-detects sibling label and GHG directories; use `labels_path=` and `ghg_path=` only when they live elsewhere.

## Local layout details and caveats

REX3 is always parsed as the complete 189-region MRIO. Region slicing is intentionally unsupported because excluding trade blocks before calculating output would produce an economically open and internally inconsistent table.

This also makes REX3 considerably larger than the other examples in this section:

- the transaction matrix has 30,807 × 30,807 values;
- the dense transaction matrix alone occupies about 7.1 GiB;
- the 1995 download is larger than 3 GB;
- parsing requires additional memory for final demand, extensions, and MARIO overhead.

Use `calc_all=False` for the initial import unless the machine has enough memory to materialize the derived matrices as well.

## Separate version directories

Both pinned source records can live under one root:

```text
REX3/
├── Version REX3 database/
│   ├── REX3_2022/
│   └── REX3_Labels/
└── Version GHG extension (+LULUCF)/
    └── GHG_data_REX3/
```

Extract `REX3_2022.zip`, `REX3_Labels.zip` and `GHG_data_REX3.zip` before local
parsing. Point `REX3_ROOT` at their common root. Retain the source labels and
notices included with the two [economic](https://zenodo.org/records/10354283)
and [GHG](https://zenodo.org/records/14724086) releases. Later record versions
are not substituted automatically.


In [ ]:
from pathlib import Path
import pandas as pd
import mario

YEAR = 2022
REX3_ROOT = Path("/path/to/REX3")
DOWNLOAD = False  # True only when MARIO should fetch the source files.

## Optional download step

If you do not already have the selected year locally, MARIO can download the yearly monetary table, the shared labels, and the complete GHG companion. Existing extracted directories are reused on subsequent runs.

In [ ]:
# Skip downloads for manually extracted economic/GHG version directories.
if DOWNLOAD:
    download_info = mario.download_exiobase_rex(REX3_ROOT, years=YEAR)
    print(download_info)


## Parse the REX3 monetary IOT

The generic EXIOBASE dispatcher is the recommended entry point. The default reads locally extracted files. Set `DOWNLOAD=True` above only when MARIO should download and extract them first.

In [ ]:
db = mario.parse_exiobase(
    path=REX3_ROOT,
    table="IOT",
    unit="Monetary",
    variant="REX3",
    year=YEAR,
    calc_all=False,
)


The direct REX3 entry point is equivalent:

```python
db = mario.parse_exiobase_rex(
    path=REX3_ROOT,
    year=YEAR,
    calc_all=False,
)
```

The default reads existing extracted files with `download=False`.

## Inspect the parsed database

Once parsed, the result is a standard MARIO database. From here the normal exploration methods apply.

A simple way to inspect it is to evaluate the object name in a notebook cell. This prints a compact summary of its core properties.

In [ ]:
db

The usual MARIO set accessors can then be used to inspect regions, sectors, consumption categories, factors of production, and satellite accounts.

In [ ]:
db.get_index("Region")[:10]

In [ ]:
db.get_index("Sector")[:10]

## Inspect the environmental extensions

The parser keeps the 17 environmental rows included in the monetary archive and adds six detailed GHG rows. Coal, oil, gas, biogenic, and calcination flows are calculated from the published `d_CC` coefficients and total REX3 output before the MARIO database is created.

LULUCF is instead read from the published absolute-flow matrices. This preserves emissions assigned to sectors whose monetary output is zero. The resulting database contains 23 satellite-account rows in total.

In [ ]:
GHG_ROWS = [
    "GHG emissions from coal",
    "GHG emissions from oil",
    "GHG emissions from gas",
    "GHG emissions from biogenic sources",
    "GHG emissions from calcination",
    "Land use change climate impacts (Blue model)",
]

E = db["baseline"]["E"]
EY = db["baseline"]["EY"]
db.get_index("Satellite account")

In [ ]:
db.units["Satellite account"].loc[GHG_ROWS]

## Check greenhouse-gas totals

The next cells aggregate the six detailed rows by country and sector. These are useful smoke tests for label alignment and implausible totals. They only operate on `E`, so they are inexpensive compared with parsing the transaction matrix.

In [ ]:
detailed_ghg = E.loc[GHG_ROWS]

ghg_by_country = (
    detailed_ghg.T
    .groupby(level="Region", sort=False)
    .sum()
    .T
)

ghg_by_country.sum(axis=0).sort_values(ascending=False).head(15).to_frame(
    "kg CO2-eq"
)

In [ ]:
ghg_by_sector = (
    detailed_ghg.T
    .groupby(level="Item", sort=False)
    .sum()
    .T
)

ghg_by_sector.sum(axis=0).sort_values(ascending=False).head(15).to_frame(
    "kg CO2-eq"
)

### Compare detailed and aggregate climate-change flows

The original aggregate `Climate change impacts` row is intentionally retained. The five detailed combustion and calcination rows are not expected to reproduce it exactly because the companion dataset omits some gases and applies aggregate mappings. LULUCF is reported separately.

The pinned 2022 source files have a global supply/uses output difference of about 0.118%, and the five detailed production GHG sources total about 1.996% less than the native climate account. Local differences can be much larger. MARIO preserves these published data without balancing or reconciling them.

In [ ]:
base_climate_total = float(E.loc["Climate change impacts"].sum())
five_source_total = float(E.loc[GHG_ROWS[:5]].to_numpy().sum())
lulucf_total = float(E.loc[GHG_ROWS[5]].sum() + EY.loc[GHG_ROWS[5]].sum())

pd.Series(
    {
        "original aggregate climate row": base_climate_total,
        "five detailed source rows": five_source_total,
        "published LULUCF, E + EY": lulucf_total,
        "five-source difference from aggregate (%)": 100
        * (five_source_total / base_climate_total - 1),
    }
)

## Notes

- prefer `mario.parse_exiobase(..., variant="REX3")` when you want one stable EXIOBASE entry point;
- use `mario.parse_exiobase_rex(...)` when the REX3-specific workflow is clearer;
- the environmental companion is always imported;
- region slicing is not supported;
- keep `calc_all=False` during the initial parse when memory is limited;
- published absolute LULUCF flows are retained, including positive flows attached to zero-output sectors.

## Sum detailed GHG sources once

The `rex3` profile sums the six detailed sources, including published LULUCF.
It excludes the overlapping native aggregate climate-change row. These are
already characterised CO2eq accounts: changing `ipcc_report` or `time_horizon`
does not reconstruct their gas inventory. Use an explicit label for the sum.
The parser uses a solve/direct preset with optional UMFPACK and SciPy fallback.
Standard EXIOBASE concordances are not assumed to apply to REX3.

The companion has no source-specific final-demand split for the first five detailed sources, whose `EY` entries are zero. Native direct household climate impacts remain in the original `Climate change impacts` row of `EY`. The detailed aggregate includes published LULUCF final-demand flows but excludes that native household climate row. Add the latter once when constructing a total that includes direct household emissions.


In [ ]:
db.calc_ghg(profile="rex3", label="GHG detailed sources")
